# Tutorial 1: Scalars and the Finite Field


**Audience:** Python developers new to cryptographic math

**Prerequisites:** Basic Python (integers, operators)

**Learning goals:**
- Understand modular arithmetic in `Z_Q`
- Use the Scalar class
- Verify field properties

## What this notebook covers

1. The curve order Q and why it matters
2. Creating Scalar objects (automatic mod Q reduction)
3. Field arithmetic: addition, subtraction, multiplication
4. Modular inverse and negation
5. Edge cases and random scalar generation
6. Exercise: verify the distributive property


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Scalar, Q


## The Curve Order Q

In secp256k1, Q is the order of the cyclic group generated by the base point G.
All private keys, nonces, and signature components are integers modulo Q. This
makes `Z_Q` a finite field: a number system where addition, subtraction,
multiplication, and division (by nonzero elements) all work, with results always
landing back in the range [0, Q).

In [ ]:
print(f"Q = {Q}")
print(f"Q has {Q.bit_length()} bits")
print(f"Q ≈ 2^{Q.bit_length()}")


## Creating Scalars

A Scalar wraps an integer and automatically reduces it modulo Q.
Any integer, no matter how large, becomes an element of `Z_Q`.

In [ ]:
a = Scalar(42)
print(f"Scalar(42) = {int(a)}")

overflow = Scalar(Q + 5)
print(f"Scalar(Q + 5) = {int(overflow)}")
print(f"Same as Scalar(5)? {int(overflow) == 5}")


## Arithmetic

Scalars support addition, subtraction, and multiplication. All operations
automatically reduce modulo Q, keeping results in the field.


In [ ]:
a = Scalar(7)
b = Scalar(11)
print(f"a + b = {int(a + b)}")
print(f"a - b = {int(a - b)}")   # wraps around: (7 - 11) mod Q
print(f"a * b = {int(a * b)}")


## Modular Inverse

Every nonzero element of `Z_Q` has a multiplicative inverse: a number a⁻¹ such
that a · a⁻¹ = 1. This is what makes `Z_Q` a *field*, not just a ring. The
implementation uses Fermat's little theorem: a⁻¹ = `a^(Q-2)` (mod Q).

In [ ]:
a = Scalar(42)
a_inv = a.inv()
product = a * a_inv
print(f"a = {int(a)}")
print(f"a⁻¹ = {int(a_inv)}")
print(f"a · a⁻¹ = {int(product)}")  # Should be 1


## Negation

The additive inverse -a is the value such that a + (-a) = 0.
In modular arithmetic, -a = Q - a.


In [ ]:
a = Scalar(42)
neg_a = -a
print(f"a = {int(a)}")
print(f"-a = {int(neg_a)}")
print(f"a + (-a) = {int(a + neg_a)}")  # Should be 0
print(f"-a is Q - a? {int(neg_a) == Q - 42}")


## Edge Cases

The scalar 0 is the additive identity, and Q wraps to 0 (since Q mod Q = 0).
The largest valid scalar is Q - 1.


In [ ]:
print(f"Scalar(0) = {int(Scalar(0))}")
print(f"Scalar(Q) = {int(Scalar(Q))}")  # Wraps to 0
print(f"Scalar(Q - 1) = {int(Scalar(Q - 1))}")  # Largest valid scalar


## Random Scalars

Cryptographic operations require random scalars (for private keys, nonces,
polynomial coefficients). `Scalar.random()` generates a cryptographically
secure random scalar using Python's `secrets` module.


In [ ]:
r = Scalar.random()
print(f"Random scalar: {int(r)}")
print(f"In range [0, Q)? {0 <= int(r) < Q}")


## Exercise

Verify the distributive property: (a + b) · c == a·c + b·c
for three random scalars. This is one of the field axioms that makes
`Z_Q` suitable for cryptographic protocols.

In [ ]:
a, b, c = Scalar.random(), Scalar.random(), Scalar.random()
lhs = (a + b) * c
rhs = a * c + b * c
print(f"(a + b) · c = {int(lhs)}")
print(f"a·c + b·c   = {int(rhs)}")
print(f"Equal? {int(lhs) == int(rhs)}")


**Try it yourself:** Verify the multiplicative identity:
a · Scalar(1) == a for a random scalar.

In [ ]:
# YOUR CODE HERE
# 1. Generate a random scalar a
# 2. Compute a * Scalar(1)
# 3. Check if the result equals a

## Pitfalls and Extensions

**Pitfall:** Python integers have arbitrary precision, so overflow isn't
an issue. In C/Rust, 256-bit arithmetic requires big-integer libraries
or careful limb management.

**Extension:** Try verifying other field axioms:
- Commutativity: a + b == b + a, a · b == b · a
- Associativity: (a + b) + c == a + (b + c)
- Multiplicative identity: a · Scalar(1) == a
